# MA-GRPO: Multi-Agent Group Relative Policy Optimization
### Training Pipeline for Colab Pro (NVIDIA A100 40GB / 80GB)

Notebook này cấu hình sẵn toàn bộ pipeline huấn luyện **Writing Agent Policy** theo đề xuất **MA-GRPO**:
- **Base Policy**: `Qwen/Qwen2.5-7B-Instruct` (QLoRA 4-bit NF4, LoRA $r=32, \alpha=64$)
- **Rollout**: $G=4$ ứng viên/prompt, độ dài $1.200 - 1.500$ tokens
- **Hardware**: Single GPU A100 (chứa cả Policy và Reward Engine trên `cuda:0` nhờ 40GB VRAM)
- **Tối ưu hóa**: FlashAttention-2 + bfloat16 native
- **Hàm thưởng nội tại**: 3 thành phần ($R_{\text{length}}$, $R_{\text{coherence}}$, $R_{\text{quality}}$)

## Bước 1: Kiểm tra môi trường GPU A100

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"BF16 Support: {bf16_ok}")
    if "A100" not in gpu_name:
        print("⚠️ Lưu ý: Bạn chưa kết nối GPU A100. Hãy vào 'Thời gian chạy' -> 'Thay đổi loại thời gian chạy' -> chọn A100.")
    else:
        print("✅ GPU A100 sẵn sàng!")

## Bước 2: Kết nối Google Drive để lưu Checkpoint
*Lưu checkpoint vào Drive để không bị mất mô hình khi hết phiên chạy.*

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/MA-GRPO-Checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Checkpoint directory ready: {DRIVE_DIR}")

## Bước 3: Nạp mã nguồn & Dataset `MA-GRPO` từ GitHub Private Repo
*Nhập GitHub Personal Access Token (PAT) để clone trực tiếp repo private.*

In [ ]:
import os
import getpass

REPO_URL = "github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"

# 1. Lấy token từ Colab Secrets (nếu đã lưu) hoặc nhập trực tiếp khi chạy
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not token:
    token = getpass.getpass("Nhập GitHub Personal Access Token của bạn: ")

# 2. Clone mã nguồn về /content/MA-GRPO
!rm -rf /content/MA-GRPO
!git clone https://{token}@{REPO_URL} /content/MA-GRPO

%cd /content/MA-GRPO
!git status
!ls -la

## Bước 4: Cài đặt các thư viện cần thiết (Tối ưu bằng PyTorch SDPA, không cần compile flash-attn)

In [ ]:
# Cài đặt các package cốt lõi
!pip install -q --upgrade pip
!pip install -q transformers>=4.45.0 accelerate>=0.33.0 peft>=0.12.0 trl>=0.15.0 datasets>=2.20.0 bitsandbytes>=0.43.0
!pip install -q sentence-transformers>=3.0.0 nltk rouge-score pydantic pyyaml pandas scipy
# Lưu ý: PyTorch 2.x SDPA đã tự động kích hoạt FlashAttention-2 bản địa trên A100, không cần compile flash-attn!

## Bước 5: Kiểm tra Config `colab_a100.yaml` và Dataset

In [ ]:
import yaml
import json

with open("configs/colab_a100.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)
print("=== CONFIG FILE ===")
print(yaml.dump(config, default_flow_style=False))

with open("data/chapters.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())
print("=== SAMPLE DATASET RECORD ===")
print(f"Target words: {sample.get('target_words')}")
print(f"Prompt preview:\n{sample.get('prompt')[:300]}...")

## Bước 6: Khởi chạy huấn luyện MA-GRPO Writing Agent
- Chạy 500 steps với nhóm $G=4$ rollouts.
- Trên GPU A100, `--reward-device cuda:0` (chạy chung trên cùng GPU vì VRAM 40GB dư dả).
- Dự kiến thời gian: **~1.8 - 2.2 giờ** (tiêu tốn ~25 CUs).

In [ ]:
!python scripts/train.py \
    --config configs/colab_a100.yaml \
    --dataset data/chapters.jsonl \
    --reward-device cuda:0 \
    --output-dir /content/drive/MyDrive/MA-GRPO-Checkpoints/ma-grpo-final \
    --max-steps 500 \
    --batch-size 1 \
    --gradient-accumulation 2 \
    --group-size 4 \
    --max-completion-tokens 1500

## Bước 7: Kiểm tra & Sinh thử chương văn bản (Inference Test)

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint_dir = "/content/drive/MyDrive/MA-GRPO-Checkpoints/ma-grpo-final"
base_model_name = "Qwen/Qwen2.5-7B-Instruct"

print("Loading fine-tuned LoRA model...")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, checkpoint_dir)
model.eval()

# Thử nghiệm sinh với prompt mẫu
inputs = tokenizer(sample["prompt"], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1500,
        temperature=0.85,
        top_p=0.95,
        repetition_penalty=1.05
    )

raw_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== GENERATED OUTPUT ===")
print(raw_output)